In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
sys.path.append("C:/Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils")

from NN_utils import *
import torch
import torch.nn as nn
from torchvision import datasets, transforms
#from torchsummary import summary
import time
from types import SimpleNamespace
import pickle
import gc
import plotly.express as px
from optimization_algorithms import *

In [3]:
transform_data = transforms.Compose([
    transforms.ToTensor()
    #transforms.Normalize((0.2868,), (0.3524,))
])

MNIST_train = datasets.MNIST(root='./data', train=True, transform=transform_data, download=True)
MNIST_test = datasets.MNIST(root='./data', train=False, transform=transform_data, download=True)

train_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_train, batch_size=60000, shuffle=True)
test_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_test, batch_size=10000, shuffle=False)

X_train_MNIST, Y_train_MNIST = next(iter(train_loader_MNIST))
X_test_MNIST, Y_test_MNIST = next(iter(test_loader_MNIST))

In [4]:
class Custom_dataset(torch.utils.data.Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [8]:
class PCA_analysis:
    def __init__(self, data_mat):
        X = data_mat.astype(np.float32, copy=False)
        self.mean_ = X.mean(axis=0, keepdims=True)
        self.std_  = X.std(axis=0, ddof=0, keepdims=True)
        X0 = (X - self.mean_) / np.where(self.std_ == 0, 1, self.std_)
        n = X0.shape[0]
        C = (X0.T @ X0) / (n - 1)
        U, S, Vh = np.linalg.svd(C, full_matrices=False)
        self.V = U
        self.S = S
        self.var_ratio_ = S / S.sum()
        self.cum_var_pct_ = np.cumsum(self.var_ratio_) * 100.0

    def compress_data(self, data_mat, k):
        """Reconstruct data using exactly k principal components."""
        X = data_mat.astype(np.float32, copy=False)
        X0 = (X - self.mean_) / np.where(self.std_ == 0, 1, self.std_)
        Vk = self.V[:, :k]
        Z = X0 @ Vk
        X0_hat = Z @ Vk.T
        return X0_hat * np.where(self.std_ == 0, 1, self.std_) + self.mean_
    
    
    def transform(self, data_mat, k):
        X = data_mat.astype(np.float32, copy=False)
        X0 = (X - self.mean_) / np.where(self.std_ == 0, 1, self.std_)
        Vk = self.V[:, :k]              # (784, k)
        Z = X0 @ Vk                     # (n_samples, k)
        return Z


In [ ]:
I_test = X_test_MNIST[:,:,:,:].detach().cpu().squeeze().numpy()
I_train = X_train_MNIST[:,:,:,:].detach().cpu().squeeze().numpy()

I_test = np.reshape(I_test,[I_test.shape[0],-1])
I_train = np.reshape(I_train,[I_train.shape[0],-1])


In [39]:
num_components_vec = [1,2,5,10,20,30,50,75,100,200,250,300,400,500,650,784]
reconst_error = []
for num_components in num_components_vec:
    
    pca_train = PCA_analysis(I_train)
    I_compressed_train = pca_train.compress_data(I_train,num_components)

    mse = np.mean((I_compressed_train - I_train)**2)
    reconst_error.append(mse)

In [40]:
fig = px.line(x=num_components_vec,y=reconst_error,markers = True,log_x=False,log_y=False)
fig.show()

In [ ]:
pca_train = PCA_analysis(I_train)
I_compressed_train = pca_train.compress_data(I_train,num_components)

pca_test = PCA_analysis(I_test)
I_compressed_test = pca_test.compress_data(I_test,num_components)

(60000, 90)

In [38]:
fig = px.line(pca_train.cum_var_pct_,log_x=False,log_y=False)
fig.show()

### Train on PCA data

In [15]:
num_components_vec = [1,2,5,10,20,30,50,75,100,200,250,300,500,784]
results = []
n_epochs = 100

stats = 1

start_time = time.time()
for s in range(stats):
    for num_components in num_components_vec:
        
        pca_train = PCA_analysis(I_train)
        I_compressed_train = pca_train.compress_data(I_train,num_components)
        X_train_MNIST = pca_train.transform(I_train,num_components)
        
        X_test_mnist = pca_train.transform(I_test,num_components)
        
        MNIST_train = Custom_dataset(X_train_MNIST,Y_train_MNIST)
        MNIST_test = Custom_dataset(X_test_mnist,Y_test_MNIST)
        
        
        train_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_train, batch_size=1000, shuffle=True)
        test_loader_MNIST = torch.utils.data.DataLoader(dataset=MNIST_test, batch_size=10000, shuffle=False)
        
        
        model_params = {
            "N_in" : num_components,
            "N_out" : 10
        }
        
        
        model = Linear_model(model_params)
        
        loss = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            
        print(f'Using {num_components} principal components, run {s+1}, number of parameters {model.count_parameters()}')
        D = train_BP_torch(model, n_epochs, train_loader_MNIST, test_loader_MNIST, loss, optimizer)
        
        results.append(D)
    
    

Using 1 principal components, run 1, number of parameters 20
Using cuda device
Linear_model(
  (W): Linear(in_features=1, out_features=10, bias=True)
)
Epoch [1/100], Step [1/60], Loss: 3.638991117477417, Test Accuracy: 15.96%
Epoch [1/100], Step [10/60], Loss: 3.411792755126953, Test Accuracy: 15.96%
Epoch [1/100], Step [20/60], Loss: 3.5041871070861816, Test Accuracy: 15.96%
Epoch [1/100], Step [30/60], Loss: 3.337148904800415, Test Accuracy: 15.96%
Epoch [1/100], Step [40/60], Loss: 3.613243341445923, Test Accuracy: 15.94%
Epoch [1/100], Step [50/60], Loss: 3.422548532485962, Test Accuracy: 16.05%
Epoch [1/100], Step [60/60], Loss: 3.2443747520446777, Test Accuracy: 16.05%
Epoch [2/100], Step [1/60], Loss: 3.2607343196868896, Test Accuracy: 16.04%
Epoch [2/100], Step [10/60], Loss: 3.345883846282959, Test Accuracy: 16.04%
Epoch [2/100], Step [20/60], Loss: 3.224820852279663, Test Accuracy: 16.04%
Epoch [2/100], Step [30/60], Loss: 3.2078726291656494, Test Accuracy: 16.04%
Epoch [2/1

In [16]:
test_loss_vec = []
train_loss_vec = []
test_acc_vec = []
for i in range(len(num_components_vec)):
    
    D = results[i]
    test_loss_vec.append(np.min(D['test_loss']))
    train_loss_vec.append(np.min(D['train_loss']))
    test_acc_vec.append(np.min(D['test_acc']))

In [28]:

fig = px.line(x=num_components_vec, y=test_loss_vec, markers=True, log_x=True)
fig.update_traces(name="Test")
fig.add_scatter(x=num_components_vec, y=train_loss_vec, mode="lines+markers", name="Train")

fig.update_layout(
    xaxis_title="Number of Principal Components",
    yaxis_title="Classification MSE",
    title="",
    width = 700,
    height = 300,
)

fig.show()

In [31]:
px.line(D['train_loss'],log_x=True,log_y = True)

AttributeError: 'PCA_analysis' object has no attribute 'Z'